# Optional — Travel Ops Toolbox SDK operations

The main [Lab 4](../labs/04-tools-toolbox.md) adds Toolbox, an OpenAPI tool, and Skills in the Portal. This notebook is an optional SDK learning aid or fallback when the UI is unavailable, not a required workshop step.

This notebook creates Travel Ops OpenAPI, Code Interpreter, Web Search, Tool Search, and connections, then invokes the Prompt Agent to inspect its answer and Conversation processing. Upload new Skills using the main Portal instructions. Updates preserve UI-added Skills, individual tool settings, guardrails, and Tool Search. Skill use is not guaranteed: without `load_skill` or `resources/read`, do not count a Skill as used.

**Kernel:** `Python (Foundry Workshop)`

Complete `00-setup.ipynb` in Codespaces or the same local Dev Container before opening this notebook.

Authentication uses only the `az login` session, not API keys or client secrets. Run one cell at a time, from top to bottom.

## 1. Locate the repository root

Locate `.workshop/context.json` and existing Python modules regardless of the folder from which the notebook was opened.

In [ ]:
import sys
from pathlib import Path


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "scripts" / "create_toolbox.py"
        ).is_file():
            return candidate
    raise RuntimeError(
        "Repository root not found. Open the notebook in your cloned workshop repository."
    )


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.lib.workshop_runtime import WORKSHOP_ENVIRONMENT, require_current_runtime  # noqa: E402

require_current_runtime(WORKSHOP_ENVIRONMENT)
print(f"Repository: {REPO_ROOT.name}")

## 2. Load the workshop context

Read participant-specific endpoints and resource names from `.workshop/context.json`, generated by `00-setup.ipynb`. You do not need to paste values into the notebook.

In [ ]:
from scripts.lib.workshop_context import (
    build_credential,
    load_context,
    project_endpoint,
    travel_api_base_url,
    validate_workshop_context,
    workshop_output,
)

CONTEXT_PATH = REPO_ROOT / ".workshop" / "context.json"
TOOLBOX_NAME = "contoso-travel-toolbox"
TOOL_NAME = "travel_ops_api"
CONNECTION_NAME = "contoso-travel-toolbox-mcp"
AGENT_NAME = "contoso-travel-assistant"

context = validate_workshop_context(load_context(CONTEXT_PATH))
foundry_endpoint = project_endpoint(context)
project_resource_id = workshop_output(context, "foundry_project_id")
travel_api_url = travel_api_base_url(context)

print(f"Foundry project endpoint: {foundry_endpoint}")
print(f"Travel Ops API: {travel_api_url}")

## 3. Create the Foundry client

`AzureCliCredential` uses the cache from `az login` in the container Terminal.

In [ ]:
from azure.ai.projects import AIProjectClient

credential = build_credential("azure-cli")
client = AIProjectClient(endpoint=foundry_endpoint, credential=credential, allow_preview=True)

print("Foundry client created.")

## 4. Fetch the Travel Ops API OpenAPI definition

Fetch `/openapi.json` from the deployed API. If the Container App is stopped, the existing helper retries a bounded number of times.

In [ ]:
from scripts import create_toolbox

openapi_spec = create_toolbox.fetch_openapi_spec(
    travel_api_url,
    create_toolbox.DEFAULT_OPENAPI_PATH,
)
operation_ids = [
    operation["operationId"]
    for path in openapi_spec["paths"].values()
    for operation in path.values()
    if isinstance(operation, dict) and "operationId" in operation
]

print(f"OpenAPI: {openapi_spec['openapi']}")
print("Operations:", ", ".join(operation_ids))

## 5. Define the OpenAPI tool

Travel Ops API is a public mock API returning only synthetic data, so authentication is `anonymous`. For a production API, replace this with managed identity or another appropriate method.

In [ ]:
auth = create_toolbox.build_auth_details("anonymous", audience=None)
travel_ops_tool = create_toolbox.build_openapi_tool(
    tool_name=TOOL_NAME,
    spec=openapi_spec,
    auth=auth,
    description=(
        "Travel Ops API for Contoso per diem lookups, cost estimates, and preapproval simulations."
    ),
)

print(f"Tool: {travel_ops_tool.name}")

## 6. Create or update the Toolbox

Rerunning with the same configuration reuses the existing Toolbox. Only a changed definition creates a new default version. Existing Portal settings are preserved; only missing Lab 4 built-in tools and Tool Search are added. Participants do not need to manage version numbers.

In [ ]:
result = create_toolbox.ensure_toolbox(
    client,
    endpoint=foundry_endpoint,
    toolbox_name=TOOLBOX_NAME,
    desired_tool=travel_ops_tool,
)

print(f"Action: {result['action']}")
print(f"Toolbox: {result['toolbox_name']}")
toolbox_endpoint = result["endpoints"]["consumer"]
print(f"MCP endpoint: {toolbox_endpoint}")

## 7. Verify the result

Use the SDK to verify that the default Toolbox contains `travel_ops_api`, Code Interpreter, Web Search, and Tool Search. The single OpenAPI entry exposes `getHealth` / `getPerDiem` / `createTripEstimate` / `createPreapproval`. If Skills have not been uploaded, return to the main Portal instructions and add them.

In [ ]:
toolbox = client.toolboxes.get(TOOLBOX_NAME)
toolbox_version = client.toolboxes.get_version(TOOLBOX_NAME, toolbox.default_version)
tool_names = [tool.name for tool in toolbox_version.tools]
tool_types = [tool.type for tool in toolbox_version.tools]

assert TOOL_NAME in tool_names, f"{TOOL_NAME} is missing from Toolbox: {tool_names}"
assert {"code_interpreter", "web_search", "toolbox_search"} <= set(tool_types)
print("Toolbox names:", ", ".join(name or "(unnamed)" for name in tool_names))
print("Toolbox types:", ", ".join(tool_types))

## 8. Create a keyless connection for the Prompt Agent

Toolbox is exposed as an MCP endpoint. Create a connection using the project managed identity so the Prompt Agent can call it without secrets.

In [ ]:
connection = create_toolbox.ensure_toolbox_connection(
    credential=credential,
    project_resource_id=project_resource_id,
    connection_name=CONNECTION_NAME,
    toolbox_endpoint=toolbox_endpoint,
)

print(f"Connection ready: {connection['name']}")

## 9. Connect Toolbox to the Prompt Agent

Add the Toolbox MCP tool while retaining the Foundry IQ knowledge base already connected to the agent. Rerunning the same configuration reuses the existing connection.

In [ ]:
agent_result = create_toolbox.attach_toolbox_to_agent(
    client,
    agent_name=AGENT_NAME,
    connection_name=CONNECTION_NAME,
    toolbox_endpoint=toolbox_endpoint,
)

print(f"Agent: {agent_result['agent_name']}")
print(f"Version: {agent_result['agent_version']}")
print(f"Action: {agent_result['action']}")
print("Toolbox connection ready (tool calls are auto-approved).")

## 10. Invoke the Prompt Agent and inspect its answer

Create an agent-specific OpenAI client and a new Conversation, then send the same estimate request as Lab 4. Section 9 enables automatic approval for the Toolbox MCP tool, so the call returns the final answer without waiting for confirmation. This call incurs model usage charges.

In [ ]:
REQUEST = """Estimate the cost of a trip from Tokyo to New York on 2026-07-10 through
2026-07-15 for 1 traveler in business class.
Do not make a booking or run an approval simulation."""

openai_client = client.get_openai_client(agent_name=AGENT_NAME)
conversation = openai_client.conversations.create()
response = openai_client.responses.create(
    conversation=conversation.id,
    input=REQUEST,
)
answer = response.output_text.strip()
normalized_answer = answer.replace(",", "")

assert answer, "The Agent returned an empty answer."
assert "781000" in normalized_answer, f"Expected total JPY 781,000 is missing:\n{answer}"
assert "booking" in answer.lower() and "approval" in answer.lower(), (
    "The answer does not clarify that this is not a booking or approval."
)

print(f"Conversation ID: {conversation.id}")
print(f"Response ID: {response.id}")
print("\nAgent output:\n")
print(answer)

## 11. Inspect Conversation processing with the SDK

Retrieve items from the same Conversation through the Responses API and display the Agent's recorded processing order. For Tool Search, inspect `tool_search` and `call_tool`, then separately inspect the selected underlying tool, arguments, and output. Depending on the runtime, downstream calls may appear only inside `call_tool` rather than being flattened into `sample.tool_calls`; do not assume a fixed item shape.

This is processing history stored in the Conversation, not an Application Insights Trace used to inspect operation durations and similar details.

In [ ]:
import json

conversation_items = openai_client.conversations.items.list(
    conversation.id,
    order="asc",
).data
item_payloads = [item.model_dump(mode="json") for item in conversation_items]

print("Conversation processing items:\n")
for index, payload in enumerate(item_payloads, start=1):
    item_type = payload.get("type", "unknown")
    item_name = payload.get("name") or "-"
    item_status = payload.get("status") or "-"
    print(f"{index}. type={item_type}, name={item_name}, status={item_status}")

mcp_calls = [payload for payload in item_payloads if payload.get("type") == "mcp_call"]
call_names = [call.get("name") for call in mcp_calls]
assert "tool_search" in call_names, f"tool_search is missing: {call_names}"
assert "call_tool" in call_names, f"call_tool is missing: {call_names}"
assert all(call.get("status") == "completed" for call in mcp_calls)
assert all(call.get("error") is None for call in mcp_calls)

selected_calls = [call for call in mcp_calls if call.get("name") == "call_tool"]
selected_arguments = [json.loads(call["arguments"]) for call in selected_calls]
selected_json = json.dumps(selected_arguments, ensure_ascii=False)
assert "createTripEstimate" in selected_json, selected_json
irrelevant_tools = (
    "createPreapproval",
    "getHealth",
    "getPerDiem",
    "web_search",
    "code_interpreter",
)
for irrelevant in irrelevant_tools:
    assert irrelevant not in selected_json, f"An unnecessary tool was executed: {irrelevant}"
for expected in ("Tokyo", "New York", "2026-07-10", "2026-07-15", "business"):
    assert expected in selected_json, f"call_tool arguments omit {expected}"

selected_outputs = [call.get("output", "") for call in selected_calls]
output_json = json.dumps(selected_outputs, ensure_ascii=False)
assert "total_estimate" in output_json, output_json
assert "781000" in output_json, output_json

print("\nTool Search order:", " -> ".join(call_names))
print("\nSelected actual tool and arguments:")
print(json.dumps(selected_arguments, ensure_ascii=False, indent=2))
print("\ncall_tool output:")
print(json.dumps(selected_outputs, ensure_ascii=False, indent=2))

## 12. Close the clients

After inspection, close the SDK clients and credential used by the notebook. The created Conversation and Trace remain available in the Portal.

In [ ]:
openai_client.close()
client.close()
credential.close()

print("Clients closed.")